<a href="https://colab.research.google.com/github/ronggobp/Machine-Learning-Course-2026/blob/main/notebooks/week-09-cnn/09_CNN_Image_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Setup & Load CIFAR-10

In [ ]:
# Install/Update wandb untuk memastikan modul integration tersedia
!pip install -q wandb

import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt
import wandb
# Perbaikan: Alamat import terbaru untuk WandbMetricsLogger
from wandb.integration.keras import WandbMetricsLogger

# Inisialisasi W&B
run = wandb.init(project="cifar10-cnn", name="cnn-baseline")

# Load Dataset Gambar Berwarna (32x32 piksel, RGB)
print("Loading CIFAR-10 data... (Mohon tunggu)")
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

# Normalisasi: Ubah pixel 0-255 menjadi 0-1 agar konvergensi lebih stabil
train_images, test_images = train_images / 255.0, test_images / 255.0

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Data Latih: {train_images.shape}")
print("Setup selesai tanpa error!")

Visualisasi Dataset

In [ ]:
plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i])
    plt.xlabel(class_names[train_labels[i][0]])
plt.show()

# Log contoh gambar ke W&B
wandb.log({"cifar_examples": [wandb.Image(train_images[i], caption=class_names[train_labels[i][0]]) for i in range(10)]})

Membangun Arsitektur CNN

In [ ]:
model = models.Sequential([
    # Gunakan layers.Input sebagai lapisan pertama secara eksplisit
    layers.Input(shape=(32, 32, 3)),

    # Layer Konvolusi 1
    # Perhatikan: input_shape sudah dihapus dari sini
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Layer Konvolusi 2
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Layer Klasifikasi (Dense)
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10)
])

model.summary()

Training & Tracking

In [ ]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

# Train model selama 10 epoch
history = model.fit(train_images, train_labels, epochs=10,
                    validation_data=(test_images, test_labels),
                    callbacks=[WandbMetricsLogger()])

Evaluasi Akhir

In [ ]:
test_loss, test_acc = model.evaluate(test_images,  test_labels, verbose=2)
print(f'\nAkurasi pada data uji: {test_acc:.4f}')

wandb.finish()